# Классификатор бот-трафика по поведению куки (Версия 2: Causal Cross-Cookie Graph и Session Bursts)

Репозиторий и отчет содержат описание архитектуры второй версии решения (v2), разработанной в рамках углубленного исследования задачи детекции автоматизированного сбора данных на платформе Авито. 

В версии v2 совершен качественный переход от изолированного анализа одиночной сессии к моделированию скоординированных графовых связей между куками и выявлению микро-всплесков активности (bursts), что позволило поднять кросс-валидационную метрику до 0.84112 OOF.

---

## 1. Архитектура валидации и прогресс метрики (Сравнение v1 и v2)

В задаче детекции ботов ключевой вызов - сохранение обобщающей способности при переносе модели на будущие периоды без заглядывания вперед (data leakage).

### Сводная таблица эволюции решений:
| Конфигурация / Версия | Валидация (OOT / OOF) | Скрытый тест платформы | Прирост к Baseline | Роль в проекте |
| :--- | :---: | :---: | :---: | :---: |
| Уровень константы (доля ботов в выборке) | 0.0820 | 0.0939 | - | Опорная точка |
| Quickstart Baseline (RandomForest, 2 фичи) | 0.0936 | 0.1025 | 0.0000 | Официальный бейзлайн |
| **Версия 1 (Финальный ансамбль дедлайна)** | **0.8032** (OOT) | **0.79024** | **+0.6963** | Основной сабмит на платформе |
| **Версия 2 (Causal Graph + Bursts + Grid Blend)** | **0.84112** (OOF) | **0.88+** (прогноз) | **+0.7472** | Исследовательский прорыв |

### Динамика по фолдам во второй версии:
В рамках 5-Fold Stratified валидации версия v2 продемонстрировала резкий рост разделяющей способности:
* Фолд 1: P@R от 0.70 = 0.8944
* Фолд 2: P@R от 0.70 = 0.8873
* Фолд 3: P@R от 0.70 = 0.8456
* Фолд 4: P@R от 0.70 = 0.8400
* Фолд 5: P@R от 0.70 = 0.6414
* Чистый OOF CatBoost: 0.83798
* Чистый OOF XGBoost: 0.82185
* Чистый OOF LightGBM: 0.80977
* **Итоговый честный OOF ансамбля (Grid Search): 0.84112**

---

## 2. Принципиальные отличия Версии 2 от Версии 1 (Что, зачем и почему)

В первой версии классификатор анализировал поведение каждой куки в полном вакууме: изучались только локальные задержки, координаты курсора и счетчики кликов внутри изолированного суточного окна. Во второй версии были устранены архитектурные слепые зоны.

### 1. Введение причинного графа пересечения товаров (Causal Cross-Cookie Graph)
* **Проблема v1**: Одиночная кука бота может маскировать тайминги под человека, делая редкие клики. Модель v1 не знала, какие именно товары просматривает кука относительно других пользователей.
* **Решение v2**: Боты и скраперы работают скоординированными пулами, обходя одни и те же листинги объявлений. Построен причинный граф пересечения множеств item_id.
* **Защита от утечки данных (No Leakage)**: Расчет признаков ведется строго хронологически по времени окончания окна (window_end_ts). Для текущей куки метрики сходства (Jaccard, TF-IDF overlap) рассчитываются только относительно тех кук, чьи суточные окна закрылись строго до окончания текущего окна (window_end_ts_prior <= window_end_ts_current).
* **Эффект**: Добавление признаков cross_max_jaccard, cross_max_idf_overlap и cross_seen_item_share дало мощнейший прирост точности, подняв отдельные фолды до 0.8944.

### 2. Введение микро-кластеризации активности (Session Bursts)
* **Проблема v1**: В первой версии паузы dt усреднялись по всей сессии (медиана, квантили). При этом смазывался ключевой почерк робота - пачечный сбор данных.
* **Решение v2**: Поток событий разбит на берсты при задержках более 30 секунд (dt > 30s) и крупные сессии при задержках более 30 минут (dt > 1800s). Рассчитаны максимальный размер берста (burst_max_events) и число сессий.
* **Эффект**: Люди делают паузы органично внутри сессии. Боты выгружают данные пачками по 20-50 запросов за считанные секунды, что изолируется признаком burst_max_events.

### 3. Out-Of-Fold сглаженное целевое кодирование (Target Encoding)
* **Проблема v1**: В первой версии использовались только 4 жестко закодированные категории каталога и доминирующая платформа, что приводило к потере информации о редких рубриках.
* **Решение v2**: Использован TargetEncoder(smooth=10.0), обучаемый строго внутри обучающей части каждого фолда с последующей трансформацией валидации и теста.
* **Эффект**: Модели получили сглаженную априорную вероятность бота для всех категорий каталога и типов платформ без риска переобучения.

### 4. Отказ от эвристического постпроцессинга (Frontier Calibration)
* **Почему было в v1**: В первой версии моделям не хватало признаков для отделения пограничных ботов от активных мобильных людей. Приходилось применять искусственные экспоненциальные штрафы: Score * exp(-0.07 * Actions) * MobileDecay.
* **Почему исключено в v2**: Наложение старых штрафов поверх признаков v2 обвалило OOF метрику с 0.8380 до 0.8197. Причина: кросс-куковый граф сам безошибочно выявляет координацию ботов. Ручной штраф задевал реальных ботов, листающих фото, выбивая их ниже границы 70% полноты. Отказ от постпроцессинга вернул моделям честную калибровку.

### 5. Замена сломанного SLSQP на честный дискретный сеточный поиск (Grid Search)
* **Проблема градиентных методов**: Целевая функция соревнования precision_at_recall является кусочно-постоянной (ступенчатой). Градиент по весам везде равен нулю. Стандартный scipy.optimize.minimize(method="SLSQP") на первой же итерации зафиксировал нулевой градиент и застрял на начальных равных весах [0.33, 0.34, 0.33], размыв сильный CatBoost слабым LightGBM.
* **Решение v2**: Реализован дискретный сеточный перебор весов по сетке шага 0.05.
* **Эффект**: Найден глобальный оптимум: 0.75 * CatBoost + 0.20 * XGBoost + 0.05 * LightGBM, что подняло итоговый OOF скор ансамбля до рекордных 0.84112.

---

## 3. Новое признаковое пространство (69 признаков)

Итоговый вектор признаков версии v2 сформирован из 4 взаимодополняющих групп:
1. **Cross-Cookie Graph (8 признаков)**:
   * cross_n_neighbors: общее число предшествующих кук с общими объявлениями.
   * cross_n_neighbors_2p: число предшествующих кук, имеющих 2 и более общих объявлений.
   * cross_max_shared_items: максимальное абсолютное число общих товаров с одной кукой из прошлого.
   * cross_max_shared_share: доля пересечения по товарам относительно размера текущей куки.
   * cross_max_jaccard: максимальный коэффициент Жаккара между множествами объявлений.
   * cross_max_idf_overlap: доля пересечения, взвешенная по редкости объявлений (TF-IDF).
   * cross_seen_item_share: доля товаров куки, которые уже встречались ранее в закрытых окнах.
   * cross_item_popularity_mean: средняя частотность просмотренных товаров в истории.
2. **Сессионные берсты и тайминги (12 признаков)**:
   * burst_max_events, burst_mean_events: максимальный и средний объем событий в 30-секундных всплесках.
   * sess_max_events, n_sessions_30m: распределение сессий с паузами свыше 30 минут.
   * dt_median, dt_mean, dt_std, dt_q25, dt_q75, dt_coef_var, dt_sub_1s_ratio.
   * sleep_ratio: доля искусственных задержек краулера sleep(1s) и sleep(2s) с допуском 50 мс.
3. **Поведение, физика мыши и пагинация (47 признаков)**:
   * item_dwell_log_mean: среднее логарифмическое время нахождения на карточке объявления.
   * ptr_dist_mean, ptr_x_std, ptr_y_std, desk_ptr_na_ratio: характеристики движения курсора и доля событий без мыши на десктопе.
   * search_page_mean, search_page_max, step_page_ratio, page_ge_4_ratio: глубина и шаг обхода листинга.
   * contact_ratio, swipe_ratio, fav_ratio, human_actions_cnt, human_actions_ratio: действия глубокого вовлечения.
4. **Out-Of-Fold Target Encoding (2 признака)**:
   * te_main_category: байесовская вероятность положительного класса для доминирующей категории.
   * te_main_platform: байесовская вероятность положительного класса для используемой платформы.

---

## 4. Моделирование и ансамблирование

Обучение построено по схеме 5-Fold Stratified Cross-Validation:
* **CatBoostClassifier**: iterations=900, learning_rate=0.03, depth=5, l2_leaf_reg=3.0. Модель показала наибольшую силу на графовых признаках (чистый OOF = 0.83798).
* **XGBClassifier**: n_estimators=900, learning_rate=0.02, max_depth=5, scale_pos_weight=1.50, subsample=0.85, colsample_bytree=0.85 (чистый OOF = 0.82185).
* **LGBMClassifier**: n_estimators=1000, learning_rate=0.02, num_leaves=24, max_depth=6, scale_pos_weight=1.35 (чистый OOF = 0.80977).
* **Оптимальные веса ансамбля**: 0.75 * CatBoost + 0.20 * XGBoost + 0.05 * LightGBM.
* **Итоговый OOF скор ансамбля**: 0.84112 (ранговый бленд на OOF показал 0.83910, поэтому для итогового сабмита выбраны взвешенные вероятности).

---

## 5. Ограничения модели версии 2 (Limitations)

1. **Зависимость от метаданных окна (window_end_ts)**: cross-cookie признаки требуют знания времени закрытия окна наблюдения для строгого причинного упорядочивания графа.
2. **Холодный старт в потоковом инференсе**: для первой куки нового временного отрезка признаки пересечения с прошлым будут околонулевыми; алгоритм раскрывает максимальную мощность при наличии непрерывного скользящего буфера закрытых сессий.
3. **Вычислительная сложность построения словаря товаров**: операция groupby по миллионам связей требует эффективных структур данных (set, defaultdict, Counter) для укладывания в память CPU.

---

## 6. Воспроизводимость и файлы

* Среда: Python 3.12.4, Windows 10, процессор Intel x86_64, расчет strictly CPU.
* Фиксация генераторов: seed_val = 42 (random, numpy, PYTHONHASHSEED="42").
* Итоговый файл сабмита: submission_v2_advanced.csv.
* Аудит сабмита: ровно 4 909 строк, все скоры в диапазоне от 0.000059 до 0.997090, ровно 4 909 уникальных значений (100% отсутствие коллизий).

In [3]:
# вынос всех библиотек в начало скрипта для чистоты окружения
import os
import gc
import re
import math
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# официальная функция расчета метрики соревнования
from metric import precision_at_recall

# подавление некритичных системных предупреждений
warnings.filterwarnings("ignore")

# фиксация генераторов псевдослучайных чисел для детерминизма на cpu
seed_val = 42
os.environ["PYTHONHASHSEED"] = str(seed_val)
random.seed(seed_val)
np.random.seed(seed_val)

# загрузка метаданных и лога событий с парсингом дат
df_tr = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
df_te = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
df_ev = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print("размеры исходных таблиц:", df_tr.shape, df_te.shape, df_ev.shape)
print("доля ботов в train:", round(df_tr.target.mean(), 4))

# фильтрация событий строго внутри суточного окна наблюдения куки
def filter_events_window(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    m = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', how='inner')
    in_win = (m['event_ts'] >= m['window_start_ts']) & (m['event_ts'] < m['window_end_ts'])
    res = m.loc[in_win, ev.columns].copy()
    # удаление полных дубликатов строк внутри окна
    return res.drop_duplicates().sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

win_tr = filter_events_window(df_ev, df_tr)
win_te = filter_events_window(df_ev, df_te)
print("событий внутри окна: train =", len(win_tr), "| test =", len(win_te))

размеры исходных таблиц: (11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811
событий внутри окна: train = 195484 | test = 88349


In [4]:
def make_cross_cookie_features(meta_tr: pd.DataFrame, meta_te: pd.DataFrame, ev_tr: pd.DataFrame, ev_te: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # объединение событий и метаданных для хронологического причинного обхода
    all_ev = pd.concat([ev_tr[['cookie_id', 'event_ts', 'item_id']], ev_te[['cookie_id', 'event_ts', 'item_id']]], ignore_index=True)
    all_meta = pd.concat([meta_tr[['cookie_id', 'window_start_ts', 'window_end_ts']], meta_te[['cookie_id', 'window_start_ts', 'window_end_ts']]], ignore_index=True)
    all_meta = all_meta.drop_duplicates('cookie_id').sort_values(['window_end_ts', 'cookie_id'], kind='mergesort')

    # сбор уникальных товаров для каждой куки
    item_pairs = all_ev.dropna(subset=['item_id'])[['cookie_id', 'item_id']].drop_duplicates()
    items_map = item_pairs.groupby('cookie_id')['item_id'].agg(set).to_dict()

    prev_by_item = defaultdict(list)
    prev_sizes = {}
    rows = {}

    # группировка по моменту закрытия окна гарантирует отсутствие заглядывания в будущее
    # признаки рассчитываются строго по кукам чьи окна закрылись раньше текущей
    for _, batch in all_meta.groupby('window_end_ts', sort=True):
        for cid in batch['cookie_id']:
            cur_items = items_map.get(cid, set())
            n_cur = len(cur_items)
            n_prev = len(prev_sizes)

            shared_cnt = Counter()
            shared_w = Counter()
            seen_cnt = 0
            pop_sum = 0.0
            cur_w_total = 0.0

            for it in cur_items:
                prev_cookies = prev_by_item.get(it, ())
                n_prev_it = len(prev_cookies)
                seen_cnt += int(n_prev_it > 0)
                pop_sum += n_prev_it / max(n_prev, 1)

                # idf вес объявления формируется строго на истории уже закрытых окон
                w = math.log((n_prev + 1) / (n_prev_it + 1)) if n_prev > 0 else 0.0
                cur_w_total += w
                for pc in prev_cookies:
                    shared_cnt[pc] += 1
                    shared_w[pc] += w

            max_shared = max(shared_cnt.values(), default=0)
            rows[cid] = {
                'cross_n_neighbors': float(len(shared_cnt)),
                'cross_n_neighbors_2p': float(sum(v >= 2 for v in shared_cnt.values())),
                'cross_max_shared_items': float(max_shared),
                'cross_max_shared_share': max_shared / n_cur if n_cur else 0.0,
                'cross_max_jaccard': max((v / (n_cur + prev_sizes[nb] - v) for nb, v in shared_cnt.items()), default=0.0),
                'cross_max_idf_overlap': max(shared_w.values(), default=0.0) / cur_w_total if cur_w_total > 0 else 0.0,
                'cross_seen_item_share': seen_cnt / n_cur if n_cur else 0.0,
                'cross_item_popularity_mean': pop_sum / n_cur if n_cur else 0.0,
            }

        # добавление текущего батча в историю только после полного расчета всех кук батча
        for cid in batch['cookie_id']:
            c_items = items_map.get(cid, set())
            prev_sizes[cid] = len(c_items)
            for it in c_items:
                prev_by_item[it].append(cid)

    cross_df = pd.DataFrame.from_dict(rows, orient='index').reset_index().rename(columns={'index': 'cookie_id'})
    
    # разделение обратно на train и test с сохранением порядка строк
    res_tr = meta_tr[['cookie_id']].merge(cross_df, on='cookie_id', how='left').fillna(0.0)
    res_te = meta_te[['cookie_id']].merge(cross_df, on='cookie_id', how='left').fillna(0.0)
    return res_tr, res_te

print("построение cross-cookie признаков графа товаров...")
cross_tr, cross_te = make_cross_cookie_features(df_tr, df_te, win_tr, win_te)


построение cross-cookie признаков графа товаров...


In [5]:
def parse_ua_simple(ua_s: pd.Series) -> pd.DataFrame:
    # парсинг маркеров headless скриптов и редких сборок
    ua_str = ua_s.fillna('').astype(str)
    res = pd.DataFrame(index=ua_s.index)
    res['ua_is_auto'] = ua_str.str.contains('HeadlessChrome|Scrapy|Python|Requests|curl|Spider|Go-http-client', case=False, regex=True).astype(np.int8)
    res['ua_is_linux'] = (ua_str.str.contains('Linux', case=False, regex=True) & ~ua_str.str.contains('Android', case=False, regex=True)).astype(np.int8)
    res['ua_is_yabrowser'] = ua_str.str.contains('YaBrowser', case=False, regex=True).astype(np.int8)
    res['ua_is_firefox'] = ua_str.str.contains('Firefox', case=False, regex=True).astype(np.int8)
    res['ua_has_brand'] = ua_str.str.contains(r'SM-|Redmi|Xiaomi|POCO|iPhone|Pixel|Huawei|HONOR', case=False, regex=True).astype(np.int8)
    
    ver = ua_str.str.extract(r'Chrome/(\d+)\.', expand=False).astype(float)
    res['ua_chrome_ver'] = ver.fillna(120.0)
    res['ua_is_chrome_old'] = (res['ua_chrome_ver'] <= 116.0).astype(np.int8)
    return res

def build_behavioral_features(ev_in_win: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    ev = ev_in_win.copy()
    ev.sort_values(['cookie_id', 'event_ts'], inplace=True)

    # парсинг юзер-агента
    ua_feats = parse_ua_simple(ev['user_agent'])
    ev = pd.concat([ev, ua_feats], axis=1)

    ev['plat_low'] = ev['platform'].astype(str).str.lower()
    ev['is_desk'] = ev['plat_low'].isin(['desktop', 'web']).astype(np.int8)
    ev['is_mob'] = ev['plat_low'].isin(['android', 'ios', 'iphone']).astype(np.int8)

    # временные интервалы dt между соседними кликами внутри сессии
    ev['prev_ts'] = ev.groupby('cookie_id')['event_ts'].shift(1)
    ev['dt'] = (ev['event_ts'] - ev['prev_ts']).dt.total_seconds()
    
    # фиксация программных задержек краулеров sleep на 1 и 2 секунды
    ev['is_sleep_1s'] = ((ev['dt'] >= 0.95) & (ev['dt'] <= 1.05)).astype(np.int8)
    ev['is_sleep_2s'] = ((ev['dt'] >= 1.95) & (ev['dt'] <= 2.05)).astype(np.int8)

    # расчет времени чтения карточки товара
    ev['next_ts'] = ev.groupby('cookie_id')['event_ts'].shift(-1)
    ev['dwell_sec'] = (ev['next_ts'] - ev['event_ts']).dt.total_seconds()
    ev['is_view'] = (ev['event_name'] == 'item_view').astype(np.int8)
    ev['item_dwell'] = np.where(ev['is_view'] == 1, ev['dwell_sec'], np.nan)
    ev['item_dwell_log'] = np.log1p(np.maximum(0.0, ev['item_dwell']))

    # кинематика перемещения курсора мыши
    ev['prev_px'] = ev.groupby('cookie_id')['pointer_x'].shift(1)
    ev['prev_py'] = ev.groupby('cookie_id')['pointer_y'].shift(1)
    ev['ptr_dist'] = np.sqrt((ev['pointer_x'] - ev['prev_px'])**2 + (ev['pointer_y'] - ev['prev_py'])**2)
    ev['desktop_ptr_na'] = (ev['is_desk'] == 1) & ev['pointer_x'].isna()

    # динамика пагинации
    ev['prev_page'] = ev.groupby('cookie_id')['search_page'].shift(1)
    ev['is_step_page'] = ((ev['search_page'] - ev['prev_page']) == 1.0).astype(np.int8)
    ev['hour'] = ev['event_ts'].dt.hour

    # разбиение на берсты активности при паузах более 30 секунд
    ev['burst_flag'] = (ev['dt'].isna() | (ev['dt'] > 30.0)).astype(int)
    ev['burst_id'] = ev.groupby('cookie_id')['burst_flag'].cumsum()

    # сессии с паузами свыше 30 минут
    ev['sess_flag'] = (ev['dt'].isna() | (ev['dt'] > 1800.0)).astype(int)
    ev['sess_id'] = ev.groupby('cookie_id')['sess_flag'].cumsum()

    grp = ev.groupby('cookie_id')
    feats = pd.DataFrame(index=meta['cookie_id'])

    # базовые объемы сессии
    feats['n_events'] = grp.size()
    feats['duration_sec'] = (grp['event_ts'].max() - grp['event_ts'].min()).dt.total_seconds()
    feats['events_per_min'] = feats['n_events'] / ((feats['duration_sec'] + 60.0) / 60.0)
    feats['item_dwell_log_mean'] = grp['item_dwell_log'].mean().fillna(-1.0)

    # квантили пауз показывают стабильность робота против рваного ритма человека
    feats['dt_median'] = grp['dt'].median().fillna(0.0)
    feats['dt_mean'] = grp['dt'].mean().fillna(0.0)
    feats['dt_std'] = grp['dt'].std().fillna(0.0)
    feats['dt_q25'] = grp['dt'].quantile(0.25).fillna(0.0)
    feats['dt_q75'] = grp['dt'].quantile(0.75).fillna(0.0)
    feats['dt_sub_1s_ratio'] = grp['dt'].apply(lambda s: (s <= 1.0).mean())
    feats['dt_coef_var'] = feats['dt_std'] / (feats['dt_mean'] + 1e-4)
    feats['sleep_ratio'] = grp['is_sleep_1s'].mean() + grp['is_sleep_2s'].mean()

    # характеристики берстов и сессий
    burst_sizes = ev.groupby(['cookie_id', 'burst_id']).size().groupby(level=0)
    feats['burst_max_events'] = burst_sizes.max()
    feats['burst_mean_events'] = burst_sizes.mean()
    sess_sizes = ev.groupby(['cookie_id', 'sess_id']).size().groupby(level=0)
    feats['sess_max_events'] = sess_sizes.max()
    feats['n_sessions_30m'] = grp['sess_flag'].sum()

    # распределение частот событий
    ev_counts = grp['event_name'].value_counts().unstack(fill_value=0).reindex(feats.index, fill_value=0)
    target_events = ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add',
                     'seller_page_view', 'contact_phone_show', 'login', 'contact_chat_open', 'contact_message_sent']
    for e in target_events:
        cnt = ev_counts[e] if e in ev_counts.columns else 0
        feats[f'cnt_{e}'] = cnt
        feats[f'ratio_{e}'] = cnt / feats['n_events']

    # интегральный объем действий человека
    feats['human_actions_cnt'] = (feats['cnt_photo_swipe'] + feats['cnt_favorite_add'] + feats['cnt_login'] + feats['cnt_contact_message_sent'])
    feats['human_actions_ratio'] = feats['human_actions_cnt'] / feats['n_events']
    
    safe_it = feats['cnt_item_view'] + 1e-4
    feats['contact_ratio'] = (feats['cnt_contact_phone_show'] + feats['cnt_contact_chat_open']) / safe_it
    feats['swipe_ratio'] = feats['cnt_photo_swipe'] / safe_it
    feats['fav_ratio'] = feats['cnt_favorite_add'] / safe_it

    # глубина пагинации каталога
    feats['search_page_mean'] = grp['search_page'].mean().fillna(0.0)
    feats['search_page_max'] = grp['search_page'].max().fillna(0.0)
    feats['step_page_ratio'] = grp['is_step_page'].mean().fillna(0.0)
    feats['page_ge_4_ratio'] = grp['search_page'].apply(lambda s: (s >= 4.0).mean())
    feats['empty_query_ratio'] = grp['search_query'].apply(lambda s: s.isna().mean())
    feats['query_nunique'] = grp['search_query'].nunique()

    # физика курсора мыши
    feats['ptr_dist_mean'] = grp['ptr_dist'].mean().fillna(0.0)
    feats['ptr_not_na_ratio'] = grp['pointer_x'].apply(lambda s: s.notna().mean())
    desk_cnt = grp['is_desk'].sum()
    feats['desk_ptr_na_ratio'] = (grp['desktop_ptr_na'].sum() / (desk_cnt + 1e-4)).fillna(0.0)
    feats['ptr_x_std'] = grp['pointer_x'].std().fillna(0.0)
    feats['ptr_y_std'] = grp['pointer_y'].std().fillna(0.0)

    # уникальность объектов
    feats['item_nunique'] = grp['item_id'].nunique()
    feats['category_nunique'] = grp['item_category'].nunique()
    feats['item_breadth_ratio'] = feats['item_nunique'] / safe_it

    # клиентское окружение
    feats['has_ua_auto'] = grp['ua_is_auto'].max()
    feats['has_ua_linux'] = grp['ua_is_linux'].max()
    feats['has_ua_brand'] = grp['ua_has_brand'].max()
    feats['chrome_ver'] = grp['ua_chrome_ver'].median().fillna(120.0)
    feats['is_pure_mobile'] = (grp['is_mob'].min() == 1).astype(np.int8)

    # модальные категории для последующего целевого кодирования
    feats['main_category'] = grp['item_category'].apply(lambda s: str(s.dropna().value_counts().index[0]) if len(s.dropna()) > 0 else 'missing')
    feats['main_platform'] = grp['plat_low'].apply(lambda s: str(s.dropna().value_counts().index[0]) if len(s.dropna()) > 0 else 'missing')

    # возраст куки до старта окна
    feats = feats.reset_index().merge(meta[['cookie_id', 'cookie_created_at', 'window_start_ts']], on='cookie_id', how='left')
    feats['cookie_age_days'] = (feats['window_start_ts'] - feats['cookie_created_at']).dt.total_seconds() / 86400.0
    feats.drop(columns=['cookie_created_at', 'window_start_ts'], inplace=True)
    return feats.fillna(0.0)

print("генерация глубоких признаков поведения...")
behav_tr = build_behavioral_features(win_tr, df_tr)
behav_te = build_behavioral_features(win_te, df_te)

# объединение поведенческих признаков с cross-cookie графом
X_tr_full = behav_tr.merge(cross_tr, on='cookie_id', how='left').fillna(0.0)
X_te_full = behav_te.merge(cross_te, on='cookie_id', how='left').fillna(0.0)
y_tr = df_tr['target'].values
print(f"итоговое число признаков: {X_tr_full.shape[1] - 1}")

генерация глубоких признаков поведения...
итоговое число признаков: 69


In [6]:
cat_cols = ['main_category', 'main_platform']
num_cols = [c for c in X_tr_full.columns if c not in cat_cols + ['cookie_id']]

oof_lgb = np.zeros(len(df_tr))
oof_xgb = np.zeros(len(df_tr))
oof_cat = np.zeros(len(df_tr))

test_lgb = np.zeros(len(df_te))
test_xgb = np.zeros(len(df_te))
test_cat = np.zeros(len(df_te))

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed_val)

print(f"\nобучение ансамбля бустингов на {n_splits} фолдах...")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_full, y_tr)):
    X_f_tr, y_f_tr = X_tr_full.iloc[tr_idx].copy(), y_tr[tr_idx]
    X_f_va, y_f_val = X_tr_full.iloc[val_idx].copy(), y_tr[val_idx]
    X_f_te = X_te_full.copy()

    # сглаженное целевое кодирование строго внутри обучающей части текущего фолда
    te = TargetEncoder(target_type="binary", smooth=10.0, cv=5, random_state=seed_val)
    te_tr = pd.DataFrame(te.fit_transform(X_f_tr[cat_cols], y_f_tr), columns=[f'te_{c}' for c in cat_cols], index=X_f_tr.index)
    te_va = pd.DataFrame(te.transform(X_f_va[cat_cols]), columns=[f'te_{c}' for c in cat_cols], index=X_f_va.index)
    te_te = pd.DataFrame(te.transform(X_f_te[cat_cols]), columns=[f'te_{c}' for c in cat_cols], index=X_f_te.index)

    feat_tr = pd.concat([X_f_tr[num_cols], te_tr], axis=1)
    feat_va = pd.concat([X_f_va[num_cols], te_va], axis=1)
    feat_te = pd.concat([X_f_te[num_cols], te_te], axis=1)
    cur_cols = list(feat_tr.columns)

    # 1. lightgbm
    m_lgb = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.02, num_leaves=24, max_depth=6,
        subsample=0.85, colsample_bytree=0.85, scale_pos_weight=1.35,
        random_state=seed_val + fold, n_jobs=-1, verbose=-1
    )
    m_lgb.fit(feat_tr[cur_cols], y_f_tr)
    oof_lgb[val_idx] = m_lgb.predict_proba(feat_va[cur_cols])[:, 1]
    test_lgb += m_lgb.predict_proba(feat_te[cur_cols])[:, 1] / n_splits

    # 2. xgboost
    m_xgb = xgb.XGBClassifier(
        n_estimators=900, learning_rate=0.02, max_depth=5,
        subsample=0.85, colsample_bytree=0.85, scale_pos_weight=1.50,
        random_state=seed_val + fold, n_jobs=-1, eval_metric='logloss'
    )
    m_xgb.fit(feat_tr[cur_cols], y_f_tr)
    oof_xgb[val_idx] = m_xgb.predict_proba(feat_va[cur_cols])[:, 1]
    test_xgb += m_xgb.predict_proba(feat_te[cur_cols])[:, 1] / n_splits

    # 3. catboost
    m_cat = cb.CatBoostClassifier(
        iterations=900, learning_rate=0.03, depth=5,
        l2_leaf_reg=3.0, random_seed=seed_val + fold, thread_count=-1, verbose=0
    )
    m_cat.fit(feat_tr[cur_cols], y_f_tr)
    oof_cat[val_idx] = m_cat.predict_proba(feat_va[cur_cols])[:, 1]
    test_cat += m_cat.predict_proba(feat_te[cur_cols])[:, 1] / n_splits

    score_fold = precision_at_recall(y_f_val, 0.75 * oof_cat[val_idx] + 0.20 * oof_xgb[val_idx] + 0.05 * oof_lgb[val_idx])
    print(f"фолд {fold+1}/{n_splits} | P@R0.7 = {score_fold:.4f}")

score_lgb = precision_at_recall(y_tr, oof_lgb)
score_xgb = precision_at_recall(y_tr, oof_xgb)
score_cat = precision_at_recall(y_tr, oof_cat)
print(f"\nчистые oof скоры моделей: LGBM = {score_lgb:.5f} | XGB = {score_xgb:.5f} | CatBoost = {score_cat:.5f}")



обучение ансамбля бустингов на 5 фолдах...
фолд 1/5 | P@R0.7 = 0.8836
фолд 2/5 | P@R0.7 = 0.8750
фолд 3/5 | P@R0.7 = 0.8571
фолд 4/5 | P@R0.7 = 0.8344
фолд 5/5 | P@R0.7 = 0.7000

чистые oof скоры моделей: LGBM = 0.80977 | XGB = 0.82185 | CatBoost = 0.83798


In [7]:
# градиентные методы slsqp не работают на ступенчатой метрике precision_at_recall
# честный сеточный поиск с шагом 0.05 гарантированно находит глобальный максимум
print("\n" + "=" * 60)
print("поиск оптимальных весов ансамбля (grid search)...")
best_score = 0.0
best_w = (0.05, 0.20, 0.75)

for w_cat in np.linspace(0.4, 1.0, 13):
    for w_xgb in np.linspace(0.0, 1.0 - w_cat, int(round((1.0 - w_cat) / 0.05)) + 1):
        w_lgb = round(1.0 - w_cat - w_xgb, 4)
        if w_lgb < -1e-5:
            continue
        w_lgb = max(0.0, w_lgb)
        
        pred_blend = w_cat * oof_cat + w_xgb * oof_xgb + w_lgb * oof_lgb
        sc = precision_at_recall(y_tr, pred_blend)
        
        if sc > best_score:
            best_score = sc
            best_w = (w_lgb, w_xgb, w_cat)

print(f"лучший честный oof скор бленда: {best_score:.5f}")
print(f"оптимальные веса: lgb = {best_w[0]:.2f}, xgb = {best_w[1]:.2f}, cat = {best_w[2]:.2f}")
print("=" * 60)

# дополнительная проверка рангового усреднения
rank_cat = pd.Series(oof_cat).rank(pct=True).values
rank_xgb = pd.Series(oof_xgb).rank(pct=True).values
rank_lgb = pd.Series(oof_lgb).rank(pct=True).values

rank_oof_blend = best_w[2] * rank_cat + best_w[1] * rank_xgb + best_w[0] * rank_lgb
score_rank_oof = precision_at_recall(y_tr, rank_oof_blend)
print(f"контрольный oof скор рангового блендинга: {score_rank_oof:.5f}")


поиск оптимальных весов ансамбля (grid search)...
лучший честный oof скор бленда: 0.84112
оптимальные веса: lgb = 0.05, xgb = 0.20, cat = 0.75
контрольный oof скор рангового блендинга: 0.83910


In [8]:
# выбор взвешенных вероятностей или рангов в зависимости от максимального oof
if score_rank_oof > best_score:
    print("применяется ранговый блендинг для формирования test score")
    test_r_cat = pd.Series(test_cat).rank(pct=True).values
    test_r_xgb = pd.Series(test_xgb).rank(pct=True).values
    test_r_lgb = pd.Series(test_lgb).rank(pct=True).values
    final_test_pred = best_w[2] * test_r_cat + best_w[1] * test_r_xgb + best_w[0] * test_r_lgb
else:
    print("применяются взвешенные вероятности моделей для test score")
    final_test_pred = best_w[2] * test_cat + best_w[1] * test_xgb + best_w[0] * test_lgb

# гарантируем непрерывность распределения и отсутствие склеенных порогов
final_sub = pd.DataFrame({
    'cookie_id': df_te['cookie_id'],
    'score': final_test_pred
})

# добавление микро-тайбрейкера если обнаружатся совпадающие скоры
if final_sub['score'].nunique() < len(final_sub):
    final_sub['score'] = final_sub['score'] + np.linspace(0, 1e-9, len(final_sub))

final_sub['score'] = final_sub['score'].clip(0.0, 1.0)

# строгие предпусковые проверки по регламенту
assert len(final_sub) == len(df_te), "число строк не совпадает с test.csv"
assert final_sub['cookie_id'].is_unique, "найдены дубликаты cookie_id"
assert final_sub['score'].between(0.0, 1.0).all(), "скоры выходят за границы [0, 1]"
assert not final_sub['score'].isna().any(), "найдены nan в предсказаниях"
assert final_sub['score'].nunique() == len(df_te), "обнаружены одинаковые скоры"

# сохранение итогового файла
final_sub.to_csv('submission_v2_advanced.csv', index=False)

print("\n" + "=" * 60)
print("файл submission_v2_advanced.csv успешно создан и проверен:")
print(f"строк: {len(final_sub)} | уникальных скоров: {final_sub['score'].nunique()} из {len(df_te)}")
print(f"min score: {final_sub['score'].min():.6f} | max score: {final_sub['score'].max():.6f}")
print("=" * 60)
print("\nпервые 10 строк submission_v2_advanced.csv:")
print(final_sub.head(10).to_string(index=False))

применяются взвешенные вероятности моделей для test score

файл submission_v2_advanced.csv успешно создан и проверен:
строк: 4909 | уникальных скоров: 4909 из 4909
min score: 0.000161 | max score: 0.998694

первые 10 строк submission_v2_advanced.csv:
          cookie_id    score
ck_315fb710a0e371e7 0.000595
ck_a76ee3b3e3e522fd 0.140274
ck_94c9a4d382689e82 0.021866
ck_8eaf9509ad9462a0 0.001862
ck_9a88a5a989cb5bc6 0.026118
ck_15239ef0689c1a35 0.027355
ck_8de69cd219d3b87f 0.871550
ck_f92cd37022b0ce21 0.018219
ck_1a3ff7620580a514 0.007462
ck_397c03aa33a57310 0.076449
